# LeetCode 554 — Brick Wall bằng Python

Link đề: https://leetcode.com/problems/brick-wall/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        ...
```

Ghi chú: bài này nhìn qua tưởng là bài hình học ("vẽ một đường thẳng đứng"), nhưng thật ra là bài **đếm tần suất**: mỗi hàng gạch cho ta một dãy "mạch vữa" (khe giữa hai viên gạch), ta chỉ cần tìm vị trí mà **nhiều hàng có mạch vữa nhất**. Notebook này đi từ cách vét cạn theo từng tọa độ, qua cách chỉ xét các vị trí có mạch vữa, đến cách dùng bảng băm (hash table) + tổng tiền tố (prefix sum), rồi hai cách khác (sắp xếp + gom nhóm, gộp k danh sách bằng heap).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tường có `n` hàng gạch cao 1 đơn vị, các hàng có cùng tổng chiều rộng; vẽ một đường thẳng đứng từ trên xuống dưới sao cho **cắt qua ít viên gạch nhất**, và đường đó **không được nằm sát hai mép trái/phải** của tường.
2. Nắm các công cụ Python cần dùng: tổng tiền tố (`itertools.accumulate` hoặc cộng dồn bằng vòng lặp), cắt lát `row[:-1]`, `collections.Counter`, `max(..., default=0)`, `sorted` + `itertools.groupby`, `heapq`.
3. Đi từ **vét cạn theo tọa độ** → **chỉ xét các vị trí là mạch vữa** → **đếm tần suất mạch vữa bằng Counter (cách tối ưu)** → **sắp xếp rồi gom nhóm** → **gộp k danh sách bằng heap**.
4. Hiểu mấu chốt: số gạch bị cắt tại vị trí `x` = `n - (số hàng có mạch vữa tại x)`. Vì vậy đi tìm **mạch vữa xuất hiện nhiều nhất**; và phải **bỏ mạch vữa cuối cùng** (chính là mép phải của tường).
5. Tránh các lỗi kinh điển: tính cả tổng tiền tố của viên gạch cuối (mép phải), gọi `max()` trên bảng đếm rỗng, trả về số hàng "sạch" thay vì phần bù `n - max`, và duyệt từng tọa độ khi chiều rộng có thể lên tới hàng tỷ.

## 2. Đề bài & ví dụ

Trước mặt bạn là một bức tường gạch hình chữ nhật có `n` hàng. Hàng thứ `i` gồm một số viên gạch, **mọi viên cao 1 đơn vị** nhưng **chiều rộng có thể khác nhau**. Tổng chiều rộng của mọi hàng là như nhau.

Hãy vẽ một đường **thẳng đứng** từ trên xuống dưới sao cho nó cắt qua **ít viên gạch nhất**. Nếu đường thẳng đi đúng qua **cạnh** (mép) của một viên gạch thì viên gạch đó **không** bị tính là bị cắt. Bạn **không được** vẽ đường nằm sát một trong hai mép dọc ngoài cùng của tường (vì khi đó nó hiển nhiên cắt 0 viên gạch).

Cho mảng hai chiều `wall` mô tả bức tường, trả về **số viên gạch bị cắt tối thiểu**.

**Ví dụ 1**

- Input: `wall = [[1,2,2,1],[3,1,2],[1,3,2],[2,4],[3,1,2],[1,3,1,1]]`
- Output: `2`

**Ví dụ 2**

- Input: `wall = [[1],[1],[1]]`
- Output: `3`

**Ràng buộc:**

- `n == wall.length`, `1 <= n <= 10^4`
- `1 <= wall[i].length <= 10^4`
- `1 <= sum(wall[i].length) <= 2 * 10^4` (tổng số viên gạch của cả bức tường không quá 20.000)
- `sum(wall[i])` bằng nhau với mọi hàng `i`
- `1 <= wall[i][j] <= 2^31 - 1` (một viên gạch có thể rộng tới hơn 2 tỷ, tổng một hàng còn lớn hơn nữa)

Hệ quả quan trọng: chiều rộng tường có thể **cực lớn** (hàng tỷ), trong khi số viên gạch chỉ tối đa 20.000. Vì thế lời giải **không được phụ thuộc vào chiều rộng**, chỉ được phụ thuộc vào số viên gạch.

In [1]:
from typing import List
from collections import Counter
from itertools import accumulate, groupby
import heapq
import random
import time

examples = [
    ([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]], 2),
    ([[1], [1], [1]], 3),
]
for wall, expected in examples:
    print("số hàng n =", len(wall), "| tổng chiều rộng mỗi hàng:", sorted({sum(r) for r in wall}), "| kỳ vọng:", expected)

số hàng n = 6 | tổng chiều rộng mỗi hàng: [6] | kỳ vọng: 2
số hàng n = 3 | tổng chiều rộng mỗi hàng: [1] | kỳ vọng: 3


## 3. Kiến thức nền

### 3.1. Tổng tiền tố (prefix sum) của một hàng

Một hàng `[1, 2, 2, 1]` nghĩa là các viên gạch rộng 1, 2, 2, 1 xếp liền nhau từ trái sang phải. **Vị trí** của các cạnh gạch tính từ mép trái (vị trí `0`) chính là **tổng cộng dồn**: sau viên đầu là `1`, sau viên thứ hai là `1+2 = 3`, sau viên thứ ba là `3+2 = 5`, sau viên cuối là `5+1 = 6` (mép phải của tường). Dãy cộng dồn này gọi là **tổng tiền tố**.

Trong Python có thể cộng dồn bằng vòng lặp (`s += x`) hoặc dùng `itertools.accumulate` (trả về iterator, bọc `list(...)` để xem).

In [2]:
row = [1, 2, 2, 1]

# Cách 1: tự cộng dồn bằng vòng lặp
s = 0
prefix = []
for x in row:
    s += x
    prefix.append(s)
print("vòng lặp  :", prefix)

# Cách 2: itertools.accumulate
print("accumulate:", list(accumulate(row)))

vòng lặp  : [1, 3, 5, 6]
accumulate: [1, 3, 5, 6]


### 3.2. Cắt lát `row[:-1]` — bỏ phần tử cuối

`row[:-1]` là bản sao của `row` **không có phần tử cuối**. Vì tổng tiền tố của **viên cuối cùng** chính là mép phải của tường (nằm sát mép, không được chọn), ta chỉ cộng dồn qua `row[:-1]`. Nếu hàng chỉ có 1 viên, `row[:-1]` là danh sách rỗng, nghĩa là hàng đó **không có mạch vữa nào** ở bên trong.

In [3]:
print([1, 2, 2, 1][:-1])       # [1, 2, 2]
print([5][:-1])                # []  (hàng chỉ có 1 viên: không có mạch vữa bên trong)
print(list(accumulate([1, 2, 2, 1][:-1])))   # [1, 3, 5]  : các mạch vữa bên trong

[1, 2, 2]
[]
[1, 3, 5]


### 3.3. `collections.Counter` — bảng đếm tần suất

`Counter` là một loại từ điển (dictionary) chuyên dùng để **đếm**: khóa là giá trị, giá trị là số lần xuất hiện. Khóa chưa có thì mặc định đếm `0`, nên `cnt[k] += 1` luôn chạy được mà không cần kiểm tra `if k in cnt`. Các hàm hay dùng: `cnt.values()`, `cnt.items()`, `cnt.most_common(1)` (cặp xuất hiện nhiều nhất).

In [4]:
cnt = Counter()
for v in [3, 1, 3, 3, 1, 7]:
    cnt[v] += 1
print(cnt)                  # Counter({3: 3, 1: 2, 7: 1})
print(cnt[3], cnt[99])      # khóa vắng mặt cho 0, không báo lỗi
print(cnt.most_common(1))   # [(3, 3)]

Counter({3: 3, 1: 2, 7: 1})
3 0
[(3, 3)]


### 3.4. `max(..., default=...)` với dãy rỗng

`max([])` báo lỗi `ValueError` vì không có phần tử nào để so. Tham số `default=` cho giá trị trả về khi dãy rỗng. Điều này rất quan trọng ở bài này: nếu **mọi hàng chỉ có 1 viên gạch** thì bảng đếm rỗng (không có mạch vữa nào), và ta muốn `max` cho `0`.

In [5]:
try:
    max([])
except ValueError as e:
    print("ValueError:", e)
print(max([], default=0))        # 0
print(max([4, 9, 2], default=0)) # 9
print(max(Counter().values(), default=0))   # bảng đếm rỗng -> 0

ValueError: max() iterable argument is empty
0
9
0


### 3.5. `sorted` + `itertools.groupby` — gom các giá trị bằng nhau

Sau khi sắp xếp, các giá trị bằng nhau nằm **cạnh nhau**. `groupby(iterable)` chia dãy thành từng "cụm" liên tiếp có cùng giá trị; với mỗi cụm nó trả về `(giá_trị, iterator_các_phần_tử)`. Đếm độ dài mỗi cụm cho ta tần suất (cách 4 sẽ dùng).

In [6]:
a = sorted([3, 1, 3, 3, 1, 7])
print(a)
for value, group in groupby(a):
    print(value, "xuất hiện", len(list(group)), "lần")

[1, 1, 3, 3, 3, 7]
1 xuất hiện 2 lần
3 xuất hiện 3 lần
7 xuất hiện 1 lần


### 3.6. `heapq` — hàng đợi ưu tiên (min-heap)

`heapq` giữ một danh sách luôn thoả: `heap[0]` là phần tử **nhỏ nhất**. `heappush(h, x)` thêm vào, `heappop(h)` lấy phần tử nhỏ nhất ra. Với tuple, Python so sánh theo thứ tự từng thành phần. Cách 5 dùng heap để "gộp" nhiều danh sách đã sắp xếp (mỗi hàng là một danh sách tăng dần các vị trí mạch vữa).

In [7]:
h = []
for x in [5, 1, 4, 1, 3]:
    heapq.heappush(h, x)
print("nhỏ nhất:", h[0])
out = []
while h:
    out.append(heapq.heappop(h))
print("lấy ra lần lượt:", out)

h2 = []
heapq.heappush(h2, (3, "hàng B"))
heapq.heappush(h2, (1, "hàng A"))
print(heapq.heappop(h2))      # (1, 'hàng A') : so sánh theo thành phần đầu

nhỏ nhất: 1
lấy ra lần lượt: [1, 1, 3, 4, 5]
(1, 'hàng A')


## 4. Ý tưởng nền tảng / trực giác

Hãy vẽ bức tường của ví dụ 1 ra (mỗi đơn vị rộng 3 ký tự, `[===]` là một viên gạch):

In [8]:
def draw_wall(wall):
    W = sum(wall[0])
    print("vị trí:", "".join(str(x).ljust(3) for x in range(W + 1)))
    for r, row in enumerate(wall):
        line = "".join("[" + "=" * (3 * w - 2) + "]" for w in row)
        print(f"hàng {r}:", line)

wall1 = examples[0][0]
draw_wall(wall1)

vị trí: 0  1  2  3  4  5  6  
hàng 0: [=][====][====][=]
hàng 1: [=======][=][====]
hàng 2: [=][=======][====]
hàng 3: [====][==========]
hàng 4: [=======][=][====]
hàng 5: [=][=======][=][=]


Nhìn hình: một đường thẳng đứng tại vị trí `x` đi qua **tất cả `n` hàng**. Ở mỗi hàng chỉ có hai khả năng:

- `x` rơi đúng vào **mạch vữa** (chỗ hai viên gạch chạm nhau) của hàng đó → hàng đó **không** bị cắt viên nào;
- `x` rơi vào **bên trong** một viên gạch → hàng đó bị cắt **đúng 1 viên**.

Vậy **số gạch bị cắt tại `x` = `n` - (số hàng có mạch vữa tại `x`)**. Muốn cắt ít gạch nhất thì phải chọn `x` có **nhiều hàng chung mạch vữa nhất**. Hãy kiểm chứng bằng cách tính tay cho từng `x = 1..5`:

In [9]:
W = sum(wall1[0])
n = len(wall1)
for x in range(1, W):
    rows_with_edge = [r for r, row in enumerate(wall1) if x in set(accumulate(row))]
    print(f"x={x}: hàng có mạch vữa = {rows_with_edge}  -> {len(rows_with_edge)} hàng; số gạch bị cắt = {n} - {len(rows_with_edge)} = {n - len(rows_with_edge)}")

x=1: hàng có mạch vữa = [0, 2, 5]  -> 3 hàng; số gạch bị cắt = 6 - 3 = 3
x=2: hàng có mạch vữa = [3]  -> 1 hàng; số gạch bị cắt = 6 - 1 = 5
x=3: hàng có mạch vữa = [0, 1, 4]  -> 3 hàng; số gạch bị cắt = 6 - 3 = 3
x=4: hàng có mạch vữa = [1, 2, 4, 5]  -> 4 hàng; số gạch bị cắt = 6 - 4 = 2
x=5: hàng có mạch vữa = [0, 5]  -> 2 hàng; số gạch bị cắt = 6 - 2 = 4


Tại `x = 4` có 4 hàng (hàng 1, 2, 4, 5) chung mạch vữa, nên chỉ bị cắt `6 - 4 = 2` viên: đúng đáp án ví dụ 1.

**Hai chú ý nhỏ:**

- `x = 0` và `x = W` (hai mép) bị cấm. Nếu cho phép, mọi hàng đều có "mạch vữa" ở đó và đáp án luôn là 0, bài toán thành vô nghĩa.
- Nếu **không có** mạch vữa bên trong nào (ví dụ mọi hàng là một viên duy nhất) thì mọi `x` đều cắt đủ `n` viên → đáp án `n` (ví dụ 2 cho `3`).

## 5. Cách 1 — Brute Force: thử từng tọa độ `x`

**Ý tưởng.** Thử **mọi** vị trí nguyên `x = 1, 2, ..., W - 1`. Với mỗi `x`, duyệt từng hàng: cộng dồn chiều rộng các viên gạch; nếu tổng tiền tố bằng đúng `x` thì `x` là mạch vữa của hàng đó (không bị cắt), nếu tổng tiền tố **vượt** `x` mà chưa từng bằng `x` thì `x` nằm trong một viên gạch (bị cắt 1 viên). Lấy `x` có số viên bị cắt nhỏ nhất.

Chú ý: nếu `W = 1` (mọi hàng là một viên rộng 1) thì không có `x` hợp lệ nào trong `1..W-1`, và đáp án là `n`.

In [10]:
def crossed_at(wall, x):
    """Số gạch bị cắt bởi đường thẳng đứng tại x (0 < x < W)."""
    crossed = 0
    for row in wall:
        s = 0
        hit_edge = False
        for w in row:
            s += w
            if s == x:
                hit_edge = True       # x là mạch vữa của hàng này
                break
            if s > x:
                break                 # x nằm trong viên gạch vừa cộng
        if not hit_edge:
            crossed += 1
    return crossed


def least_bricks_brute(wall):
    W = sum(wall[0])
    best = len(wall)                  # nếu không có x hợp lệ, mọi hàng đều bị cắt
    for x in range(1, W):
        best = min(best, crossed_at(wall, x))
    return best


for wall, expected in examples:
    got = least_bricks_brute(wall)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


**Độ phức tạp:** Time `O(W · N)` với `W` là chiều rộng tường và `N` là tổng số viên gạch (mỗi `x` duyệt qua gần hết `N` viên). Space `O(1)`.

**Nút thắt:**

1. `W` có thể lên tới hàng **tỷ** (một viên gạch đã rộng tới `2^31 - 1`), nên vòng `for x in range(1, W)` không thể chạy nổi.
2. Phần lớn các `x` **không phải mạch vữa của hàng nào** → đều cho kết quả `n`, tức là ta tốn công vô ích. Chỉ những `x` là tổng tiền tố của ít nhất một hàng mới đáng xét. Đó là bước đột phá ở mục 6.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát 1 — số ứng viên rất ít.** Tại `x` không phải mạch vữa của hàng nào, mọi hàng đều bị cắt, được `n` (là mức tệ nhất). Vì vậy chỉ cần xét các `x` là **tổng tiền tố bên trong** (không tính mép phải) của ít nhất một hàng. Số ứng viên tối đa bằng `N - n` (≤ 20.000), không phụ thuộc `W`.

**Quan sát 2 — đảo vai trò: thay vì hỏi từng `x` "có bao nhiêu hàng có mạch vữa ở đây?", ta để từng hàng "bỏ phiếu".** Mỗi hàng cộng dồn chiều rộng, mỗi lần được một tổng tiền tố bên trong thì **bỏ một phiếu** cho vị trí đó. Cuối cùng phiếu của vị trí `x` chính là số hàng có mạch vữa tại `x`. Một hàng không thể bỏ hai phiếu cho cùng một `x` vì chiều rộng gạch `≥ 1` nên tổng tiền tố trong một hàng **tăng chặt**.

**Công thức:**

```
cnt[x] = số hàng có mạch vữa tại x
đáp án = n - max(cnt.values(), default=0)
```

Mỗi viên gạch (trừ viên cuối của mỗi hàng) được xử lý đúng một lần nên tổng thời gian chỉ là `O(N)`.

## 7. Các cách tiếp theo

Phần này có thêm 4 cách nữa (tổng 5 cách): chỉ xét các vị trí là mạch vữa (set mỗi hàng), đếm bằng Counter (cách tối ưu), sắp xếp + gom nhóm, và gộp k danh sách bằng heap.

### 7.1. Cách 2 — Chỉ thử các vị trí là mạch vữa (set cho từng hàng)

**Giải thích từng đoạn.** Bước 1: với mỗi hàng, lập **tập hợp** (`set`) các mạch vữa bên trong của hàng đó. Bước 2: gom mọi mạch vữa của mọi hàng thành tập ứng viên. Bước 3: với mỗi ứng viên `x`, đếm số hàng **có** `x` trong tập của mình; số gạch bị cắt là `n - đếm`. Tra cứu `x in set` chỉ tốn `O(1)` nên mỗi ứng viên tốn `O(n)`.

In [11]:
def least_bricks_candidates(wall):
    row_sets = [set(accumulate(row[:-1])) for row in wall]    # mạch vữa bên trong mỗi hàng
    candidates = set().union(*row_sets)                       # mọi vị trí đáng thử
    best = len(wall)                                          # không có ứng viên -> n
    for x in candidates:
        has_edge = sum(1 for s in row_sets if x in s)         # số hàng có mạch vữa tại x
        best = min(best, len(wall) - has_edge)
    return best


for wall, expected in examples:
    print(least_bricks_candidates(wall), "(kỳ vọng", expected, ")")
    assert least_bricks_candidates(wall) == expected

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [12]:
class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        row_sets = [set(accumulate(row[:-1])) for row in wall]
        candidates = set().union(*row_sets)
        best = len(wall)
        for x in candidates:
            has_edge = sum(1 for s in row_sets if x in s)
            best = min(best, len(wall) - has_edge)
        return best

print(Solution().leastBricks([[1], [1], [1]]))

3


**Độ phức tạp:** Time `O(E · n)` với `E` là số ứng viên (≤ `N`) và `n` là số hàng; trường hợp xấu `E ≈ 10^4`, `n ≈ 10^4` cho ~`10^8` thao tác: dễ **quá thời gian**. Space `O(N)` cho các set. Cách này đã bỏ được sự phụ thuộc vào `W`, nhưng vẫn **hỏi lại từng hàng cho từng ứng viên** - đúng điều mà quan sát 2 (để các hàng "bỏ phiếu") giúp ta tránh.

### 7.2. Cách 3 — Hash table + prefix sum (Counter): cách tối ưu

**Giải thích từng đoạn.** Dùng một `Counter` `cnt`. Duyệt từng hàng; với mỗi hàng đặt `s = 0`, đi qua `row[:-1]`, cộng `s += x` rồi `cnt[s] += 1`. Mỗi hàng bỏ đúng một phiếu cho mỗi mạch vữa bên trong của mình. Cuối cùng, `max(cnt.values(), default=0)` là số hàng chung mạch vữa nhiều nhất, và đáp án là `n` trừ đi giá trị đó.

In [13]:
def least_bricks_counter(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:        # bỏ viên cuối: tổng của nó là mép phải của tường
            s += x
            cnt[s] += 1           # hàng này "bỏ một phiếu" cho vị trí s
    return len(wall) - max(cnt.values(), default=0)


for wall, expected in examples:
    got = least_bricks_counter(wall)
    print(got, "(kỳ vọng", expected, ")")
    assert got == expected

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [14]:
from collections import Counter


class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        cnt = Counter()
        for row in wall:
            s = 0
            for x in row[:-1]:
                s += x
                cnt[s] += 1
        return len(wall) - max(cnt.values(), default=0)


sol = Solution()
for wall, expected in examples:
    print(sol.leastBricks(wall))

2
3


**Độ phức tạp:** Time `O(N)` (mỗi viên gạch xử lý đúng một lần, thao tác `Counter` trung bình `O(1)`). Space `O(E) ⊆ O(N)` cho số khóa khác nhau trong Counter (tối đa `N - n`).

### 7.3. Cách 4 — Sắp xếp mọi mạch vữa rồi gom nhóm

**Giải thích từng đoạn.** Thay vì bảng băm, ta **gom hết mọi mạch vữa của mọi hàng vào một danh sách phẳng**, **sắp xếp**, rồi các giá trị bằng nhau sẽ nằm liền kề. Đo độ dài cụm dài nhất bằng `groupby`. Đây là cách "đếm bằng sắp xếp" quen thuộc khi không muốn dùng bảng băm (hoặc khi ngôn ngữ khiến băm kém an toàn). Giá phải trả: thêm hệ số `log`.

In [15]:
def least_bricks_sorted(wall):
    edges = []
    for row in wall:
        edges.extend(accumulate(row[:-1]))     # mọi mạch vữa bên trong của mọi hàng
    edges.sort()
    best = 0
    for _, group in groupby(edges):
        best = max(best, sum(1 for _ in group))   # độ dài cụm = số hàng chung mạch vữa
    return len(wall) - best


for wall, expected in examples:
    print(least_bricks_sorted(wall), "(kỳ vọng", expected, ")")
    assert least_bricks_sorted(wall) == expected

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [16]:
class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        edges = []
        for row in wall:
            edges.extend(accumulate(row[:-1]))
        edges.sort()
        best = 0
        for _, group in groupby(edges):
            best = max(best, sum(1 for _ in group))
        return len(wall) - best

print(Solution().leastBricks([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]))

2


**Độ phức tạp:** Time `O(N log N)` (sắp xếp tối đa `N` phần tử). Space `O(N)` cho danh sách phẳng. Chậm hơn Counter một hệ số `log N`, nhưng cho thấy bản chất bài là **tìm giá trị xuất hiện nhiều nhất** trong một dãy.

### 7.4. Cách 5 — Gộp k danh sách đã sắp xếp bằng heap (quét từ trái sang phải)

**Giải thích từng đoạn.** Mỗi hàng, tự nó, là một danh sách mạch vữa **đã tăng dần**. Ta mô phỏng "quét một đường thẳng đứng từ trái sang phải": heap chứa, với mỗi hàng, **mạch vữa kế tiếp chưa quét** dưới dạng `(vị trí, chỉ số hàng, chỉ số viên)`. Mỗi vòng lặp:

1. lấy vị trí nhỏ nhất `p` trong heap;
2. lấy ra **mọi** phần tử có vị trí đúng bằng `p` (đó là các hàng chung mạch vữa tại `p`), đếm số lượng;
3. với mỗi hàng vừa lấy ra, đẩy mạch vữa kế tiếp của hàng đó (nếu còn) vào heap.

Cách này không cần bảng đếm cho toàn bộ vị trí, heap chỉ chứa **tối đa `n` phần tử** tại mọi thời điểm, nên bộ nhớ phụ chỉ `O(n)`.

In [17]:
def least_bricks_heap(wall):
    heap = []
    for r, row in enumerate(wall):
        if len(row) >= 2:                          # hàng có ít nhất 1 mạch vữa bên trong
            heapq.heappush(heap, (row[0], r, 0))   # mạch vữa đầu tiên của hàng r
    best = 0
    while heap:
        p = heap[0][0]
        same = 0
        while heap and heap[0][0] == p:            # lấy mọi hàng chung mạch vữa tại p
            _, r, i = heapq.heappop(heap)
            same += 1
            if i + 2 < len(wall[r]):               # còn mạch vữa kế tiếp (không tính viên cuối)
                heapq.heappush(heap, (p + wall[r][i + 1], r, i + 1))
        best = max(best, same)
    return len(wall) - best


for wall, expected in examples:
    print(least_bricks_heap(wall), "(kỳ vọng", expected, ")")
    assert least_bricks_heap(wall) == expected

2 (kỳ vọng 2 )
3 (kỳ vọng 3 )


In [18]:
class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        heap = []
        for r, row in enumerate(wall):
            if len(row) >= 2:
                heapq.heappush(heap, (row[0], r, 0))
        best = 0
        while heap:
            p = heap[0][0]
            same = 0
            while heap and heap[0][0] == p:
                _, r, i = heapq.heappop(heap)
                same += 1
                if i + 2 < len(wall[r]):
                    heapq.heappush(heap, (p + wall[r][i + 1], r, i + 1))
            best = max(best, same)
        return len(wall) - best

print(Solution().leastBricks([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]))

2


**Độ phức tạp:** Time `O(N log n)` (mỗi mạch vữa vào/ra heap một lần; heap có tối đa `n` phần tử). Space `O(n)` cho heap. Phù hợp khi các danh sách đã sắp xếp sẵn và ta muốn tiết kiệm bộ nhớ; còn thực tế Counter thường nhanh nhất và ngắn nhất.

## 8. TRACE từng bước

Trace cách Counter với ví dụ 1: `wall = [[1,2,2,1],[3,1,2],[1,3,2],[2,4],[3,1,2],[1,3,1,1]]`, `n = 6`, kỳ vọng `2`.

Mạch vữa bên trong mỗi hàng (tổng tiền tố của `row[:-1]`):

| hàng | `row` | `row[:-1]` | mạch vữa bên trong | `cnt` sau hàng này (khóa tăng dần) |
|---|---|---|---|---|
| 0 | `[1,2,2,1]` | `[1,2,2]` | 1, 3, 5 | `{1:1, 3:1, 5:1}` |
| 1 | `[3,1,2]` | `[3,1]` | 3, 4 | `{1:1, 3:2, 4:1, 5:1}` |
| 2 | `[1,3,2]` | `[1,3]` | 1, 4 | `{1:2, 3:2, 4:2, 5:1}` |
| 3 | `[2,4]` | `[2]` | 2 | `{1:2, 2:1, 3:2, 4:2, 5:1}` |
| 4 | `[3,1,2]` | `[3,1]` | 3, 4 | `{1:2, 2:1, 3:3, 4:3, 5:1}` |
| 5 | `[1,3,1,1]` | `[1,3,1]` | 1, 4, 5 | `{1:3, 2:1, 3:3, 4:4, 5:2}` |

Kết quả: `max(cnt.values()) = 4` (tại `x = 4`) → đáp án `6 - 4 = 2`.

In [19]:
def trace_counter(wall):
    cnt = Counter()
    n = len(wall)
    print("n =", n)
    for r, row in enumerate(wall):
        s = 0
        votes = []
        for x in row[:-1]:
            s += x
            cnt[s] += 1
            votes.append(s)
        print(f"hàng {r}: row={row} -> bỏ phiếu cho {votes}; cnt = {dict(sorted(cnt.items()))}")
    best = max(cnt.values(), default=0)
    where = sorted(k for k, v in cnt.items() if v == best)
    print(f"max cnt = {best} (tại x = {where}) -> đáp án = {n} - {best} = {n - best}")
    return n - best

assert trace_counter(examples[0][0]) == 2

n = 6
hàng 0: row=[1, 2, 2, 1] -> bỏ phiếu cho [1, 3, 5]; cnt = {1: 1, 3: 1, 5: 1}
hàng 1: row=[3, 1, 2] -> bỏ phiếu cho [3, 4]; cnt = {1: 1, 3: 2, 4: 1, 5: 1}
hàng 2: row=[1, 3, 2] -> bỏ phiếu cho [1, 4]; cnt = {1: 2, 3: 2, 4: 2, 5: 1}
hàng 3: row=[2, 4] -> bỏ phiếu cho [2]; cnt = {1: 2, 2: 1, 3: 2, 4: 2, 5: 1}
hàng 4: row=[3, 1, 2] -> bỏ phiếu cho [3, 4]; cnt = {1: 2, 2: 1, 3: 3, 4: 3, 5: 1}
hàng 5: row=[1, 3, 1, 1] -> bỏ phiếu cho [1, 4, 5]; cnt = {1: 3, 2: 1, 3: 3, 4: 4, 5: 2}
max cnt = 4 (tại x = [4]) -> đáp án = 6 - 4 = 2


Trace ví dụ 2 `wall = [[1],[1],[1]]`: mọi hàng chỉ có 1 viên nên `row[:-1]` rỗng, không ai bỏ phiếu, `cnt` rỗng, `max(..., default=0) = 0`, đáp án `3 - 0 = 3`.

In [20]:
assert trace_counter(examples[1][0]) == 3
print()
# một tường mà các mạch vữa trùng nhau hoàn toàn: đáp án 0
assert trace_counter([[2, 3], [2, 3], [2, 3]]) == 0

n = 3
hàng 0: row=[1] -> bỏ phiếu cho []; cnt = {}
hàng 1: row=[1] -> bỏ phiếu cho []; cnt = {}
hàng 2: row=[1] -> bỏ phiếu cho []; cnt = {}
max cnt = 0 (tại x = []) -> đáp án = 3 - 0 = 3

n = 3
hàng 0: row=[2, 3] -> bỏ phiếu cho [2]; cnt = {2: 1}
hàng 1: row=[2, 3] -> bỏ phiếu cho [2]; cnt = {2: 2}
hàng 2: row=[2, 3] -> bỏ phiếu cho [2]; cnt = {2: 3}
max cnt = 3 (tại x = [2]) -> đáp án = 3 - 3 = 0


Trace cách 5 (heap) trên ví dụ 1 để thấy việc "quét" từ trái sang phải: mỗi vòng ngoài xử lý đúng một vị trí `p`.

| vòng | `p` nhỏ nhất | các hàng chung mạch vữa | số hàng | `best` |
|---|---|---|---|---|
| 1 | 1 | 0, 2, 5 | 3 | 3 |
| 2 | 2 | 3 | 1 | 3 |
| 3 | 3 | 0, 1, 4 | 3 | 3 |
| 4 | 4 | 1, 2, 4, 5 | 4 | 4 |
| 5 | 5 | 0, 5 | 2 | 4 |

In [21]:
def trace_heap(wall):
    heap = []
    for r, row in enumerate(wall):
        if len(row) >= 2:
            heapq.heappush(heap, (row[0], r, 0))
    best = 0
    rnd = 0
    while heap:
        rnd += 1
        p = heap[0][0]
        rows = []
        while heap and heap[0][0] == p:
            _, r, i = heapq.heappop(heap)
            rows.append(r)
            if i + 2 < len(wall[r]):
                heapq.heappush(heap, (p + wall[r][i + 1], r, i + 1))
        best = max(best, len(rows))
        print(f"vòng {rnd}: p={p}, hàng chung mạch vữa={sorted(rows)}, số hàng={len(rows)}, best={best}")
    print("đáp án =", len(wall) - best)
    return len(wall) - best

assert trace_heap(examples[0][0]) == 2

vòng 1: p=1, hàng chung mạch vữa=[0, 2, 5], số hàng=3, best=3
vòng 2: p=2, hàng chung mạch vữa=[3], số hàng=1, best=3
vòng 3: p=3, hàng chung mạch vữa=[0, 1, 4], số hàng=3, best=3
vòng 4: p=4, hàng chung mạch vữa=[1, 2, 4, 5], số hàng=4, best=4
vòng 5: p=5, hàng chung mạch vữa=[0, 5], số hàng=2, best=4
đáp án = 2


## 9. Code nộp LeetCode

Lời giải tối ưu (Cách 3 - Hash table + prefix sum):

In [22]:
from collections import Counter
from typing import List


class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        cnt = Counter()
        for row in wall:
            s = 0
            for x in row[:-1]:          # không tính viên cuối (mép phải của tường)
                s += x
                cnt[s] += 1
        return len(wall) - max(cnt.values(), default=0)


sol = Solution()
for wall, expected in examples:
    got = sol.leastBricks(wall)
    print(got)
    assert got == expected

2
3


## 10. Vì sao đúng

**Bổ đề 1 — công thức số gạch bị cắt.** Đường thẳng đứng tại `x` (với `0 < x < W`) đi qua đúng một viên gạch trong mỗi hàng (hoặc đi qua ranh giới giữa hai viên). Hàng `i` có mạch vữa tại `x` khi và chỉ khi `x` là một tổng tiền tố **bên trong** của hàng `i` (tức tổng của một tiền tố của `row` **không rỗng** và **không đầy đủ**). Nếu có mạch vữa thì hàng đó không bị cắt viên nào, nếu không thì bị cắt đúng 1 viên. Do đó:

```
crossed(x) = n - #{ hàng i có mạch vữa tại x }
```

**Bổ đề 2 — đếm phiếu bằng số hàng.** Chiều rộng mỗi viên `≥ 1`, nên các tổng tiền tố của một hàng **tăng chặt**, tức là không có hai tiền tố nào trong cùng một hàng cho cùng một giá trị. Vì vậy mỗi hàng bỏ **tối đa một phiếu** cho mỗi `x`, và `cnt[x]` = đúng số hàng có mạch vữa tại `x`.

**Bổ đề 3 — loại viên cuối.** Tổng tiền tố đầy đủ của mọi hàng đều bằng `W` (chiều rộng chung), tức là mép phải bị cấm. Mép trái `x = 0` tương ứng với tiền tố rỗng; vòng lặp cộng dồn bắt đầu bằng `s += x` rồi mới đếm, nên không bao giờ đếm `0`. Cắt lát `row[:-1]` loại đúng tiền tố đầy đủ.

**Bổ đề 4 — chỉ cần xét `x` có phiếu.** Nếu `x` hợp lệ nhưng `cnt[x] = 0`, thì `crossed(x) = n`, là giá trị lớn nhất có thể; bất kỳ `x` nào có phiếu đều cho giá trị `≤ n - 1`. Khi bảng đếm rỗng thì mọi `x` hợp lệ đều cho `n` (hoặc không có `x` hợp lệ khi `W = 1`), và đáp án `n - 0 = n` đúng.

**Kết luận.** `min_x crossed(x) = n - max_x cnt[x]`, với quy ước `max = 0` khi bảng đếm rỗng. Mọi cách trong notebook đều tính cùng một đại lượng nên cùng cho đáp án đúng.

In [23]:
# Kiểm chứng Bổ đề 2: tổng tiền tố trong một hàng tăng chặt, nên không có phiếu trùng trong cùng hàng
import random as _r
_r.seed(1)
for _ in range(1000):
    row = [_r.randint(1, 9) for _ in range(_r.randint(1, 8))]
    pre = list(accumulate(row[:-1]))
    assert len(pre) == len(set(pre))
    assert all(a < b for a, b in zip(pre, pre[1:]))
print("Bổ đề 2 đúng trên 1000 hàng ngẫu nhiên")

Bổ đề 2 đúng trên 1000 hàng ngẫu nhiên


## 11. Time / Space Complexity

Đặt `n` là số hàng, `N` là tổng số viên gạch của cả tường (`N ≤ 2 · 10^4`), `E` là số vị trí mạch vữa khác nhau (`E ≤ N - n`), `W` là chiều rộng tường.

| Cách | Time | Space |
|---|---|---|
| 1. Thử từng tọa độ | `O(W · N)` | `O(1)` |
| 2. Set mỗi hàng + thử các ứng viên | `O(E · n + N)` | `O(N)` |
| 3. Counter (tối ưu) | `O(N)` | `O(E)` |
| 4. Sắp xếp + groupby | `O(N log N)` | `O(N)` |
| 5. Heap gộp k danh sách | `O(N log n)` | `O(n)` |

Với Cách 3: đúng mỗi viên gạch (trừ viên cuối mỗi hàng) được xử lý một lần với thao tác băm `O(1)`, nên `O(N)`. Vì `N ≤ 2 · 10^4`, chương trình chạy thừa nhanh. **Lưu ý ngôn ngữ khác:** tổng tiền tố có thể vượt 32 bit (một hàng rộng tới `10^4 · 2^31`), nên Java/C++ phải dùng `long`/`long long`; Python có số nguyên lớn tùy ý nên không lo tràn.

In [24]:
# Đo thực nghiệm: n = 10^4 hàng, mỗi hàng 2 viên (tổng viên = 2 * 10^4), chiều rộng lên tới ~2 tỷ
rnd = random.Random(554)
W = 2 ** 31 - 1
big = []
for _ in range(10_000):
    a = rnd.choice([1, 2, 3, 1000, W - 1])
    big.append([a, W - a])

t0 = time.perf_counter()
ans_counter = least_bricks_counter(big)
t1 = time.perf_counter()
ans_sorted = least_bricks_sorted(big)
t2 = time.perf_counter()
ans_heap = least_bricks_heap(big)
t3 = time.perf_counter()
print("đáp án Counter / sorted / heap:", ans_counter, ans_sorted, ans_heap)
assert ans_counter == ans_sorted == ans_heap
print("Cả ba cách đều xong trong vài chục mili giây:", max(t1 - t0, t2 - t1, t3 - t2) < 2.0)

đáp án Counter / sorted / heap: 7957 7957 7957
Cả ba cách đều xong trong vài chục mili giây: True


## 12. Bộ test

Chia làm 4 nhóm: (a) ví dụ đề và edge case; (b) phản ví dụ cho các heuristic sai; (c) đối chiếu các cách với một **oracle độc lập** (lưới ô vuông); (d) test ngẫu nhiên có seed đúng ràng buộc.

### 12.1. Oracle độc lập và các test cố định

Oracle dựng một **lưới ô vuông**: mỗi hàng là một danh sách mã viên gạch cho từng ô đơn vị; đường thẳng ở vị trí `x` cắt viên gạch ở hàng đó khi ô `x - 1` và ô `x` **cùng mã viên**. Cách này không dùng tổng tiền tố hay đếm phiếu nên độc lập với mọi lời giải ở trên.

In [25]:
def oracle_grid(wall):
    n = len(wall)
    W = sum(wall[0])
    grid = []
    for row in wall:
        cells = []
        for brick_id, w in enumerate(row):
            cells.extend([brick_id] * w)       # viên brick_id chiếm w ô liên tiếp
        assert len(cells) == W
        grid.append(cells)
    best = n
    for x in range(1, W):                      # đường nằm giữa ô x-1 và ô x
        crossed = sum(1 for cells in grid if cells[x - 1] == cells[x])
        best = min(best, crossed)
    return best


solvers = {
    "brute": least_bricks_brute,
    "candidates": least_bricks_candidates,
    "counter": least_bricks_counter,
    "sorted": least_bricks_sorted,
    "heap": least_bricks_heap,
    "submit": Solution().leastBricks,
}

fixed_cases = [
    ([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]], 2),   # ví dụ 1
    ([[1], [1], [1]], 3),                                                       # ví dụ 2
    ([[5]], 1),                                                                 # 1 hàng, 1 viên
    ([[1, 1, 1, 1, 1]], 0),                                                     # 1 hàng nhiều viên: vẽ vào mạch vữa
    ([[2, 3], [2, 3], [2, 3]], 0),                                              # mạch vữa trùng hoàn toàn
    ([[1, 2], [2, 1]], 1),                                                      # hai mạch vữa khác nhau
    ([[3], [1, 2], [2, 1], [3]], 3),                                            # hàng 1 viên không có mạch vữa
    ([[1, 1], [1, 1], [2]], 1),                                                 # một hàng một viên
    ([[2, 2, 2], [3, 3], [6], [1, 5]], 3),                                      # nhiều ứng viên, hàng không chung
]
for wall, expected in fixed_cases:
    assert oracle_grid(wall) == expected, (wall, oracle_grid(wall), expected)
    for name, f in solvers.items():
        assert f(wall) == expected, (name, wall, f(wall), expected)
print("Các test cố định: OK (", len(fixed_cases), "bộ,", len(solvers), "lời giải + oracle )")

Các test cố định: OK ( 9 bộ, 6 lời giải + oracle )


### 12.2. Edge case về giá trị lớn

Giá trị gạch tới `2^31 - 1`, tổng tiền tố vượt 32 bit. Brute theo tọa độ và oracle lưới **không** chạy được ở cỡ này, nên chỉ so các cách dựa trên mạch vữa với đáp án tính tay.

In [26]:
M = 2 ** 31 - 1
big_cases = [
    ([[M, M], [M, M]], 0),                         # mạch vữa tại M ở cả hai hàng
    ([[M, 1], [1, M]], 1),                         # hai hàng, mạch vữa khác nhau
    ([[M, M, M], [M, M, M], [2 * M, M]], 0),       # mạch vữa 2M ở cả ba hàng
    ([[M], [M], [M]], 3),                          # mọi hàng một viên rất rộng
    ([[1, M], [M, 1], [1, M], [M, 1], [1, M]], 2),
    ([[M - 1, 1, M], [M, M], [1, M - 1, M]], 0),   # cả ba hàng đều có mạch vữa tại M
]
fast_solvers = {k: v for k, v in solvers.items() if k not in ("brute",)}
for wall, expected in big_cases:
    for name, f in fast_solvers.items():
        got = f(wall)
        assert got == expected, (name, wall, got, expected)
print("Test giá trị lớn: OK")

Test giá trị lớn: OK


### 12.3. Phản ví dụ cho các heuristic sai

Có hai heuristic nghe "hợp lý" nhưng sai:

- *"Chỉ cần xét mạch vữa của hàng đầu tiên"* (vì đường thẳng phải đi qua hàng đầu). Phản ví dụ: hàng đầu là một viên duy nhất không có mạch vữa nhưng các hàng khác lại chung mạch vữa.
- *"Chọn mạch vữa của hàng nhiều viên nhất"*. Phản ví dụ: hàng nhiều viên nhất có các mạch vữa "lạc" không trùng với ai.

In [27]:
def wrong_first_row_only(wall):
    first_edges = set(accumulate(wall[0][:-1]))
    cnt = Counter()
    for row in wall:
        for s in accumulate(row[:-1]):
            if s in first_edges:
                cnt[s] += 1
    return len(wall) - max(cnt.values(), default=0)


def wrong_longest_row(wall):
    longest = max(wall, key=len)
    edges = set(accumulate(longest[:-1]))
    cnt = Counter()
    for row in wall:
        for s in accumulate(row[:-1]):
            if s in edges:
                cnt[s] += 1
    return len(wall) - max(cnt.values(), default=0)


w1 = [[6], [1, 5], [1, 5], [2, 4]]
print("first_row_only:", wrong_first_row_only(w1), "| đúng:", Solution().leastBricks(w1))
assert wrong_first_row_only(w1) != Solution().leastBricks(w1)

w2 = [[2, 2, 2, 2], [1, 7], [1, 7], [1, 7]]
print("longest_row   :", wrong_longest_row(w2), "| đúng:", Solution().leastBricks(w2))
assert wrong_longest_row(w2) != Solution().leastBricks(w2)
assert oracle_grid(w1) == Solution().leastBricks(w1)
assert oracle_grid(w2) == Solution().leastBricks(w2)

first_row_only: 4 | đúng: 2
longest_row   : 3 | đúng: 1


### 12.4. Test ngẫu nhiên có seed, đối chiếu các cách và oracle

Sinh tường nhỏ hợp lệ: chọn `W` trong `1..14`, `n` trong `1..8`; mỗi hàng chọn ngẫu nhiên một tập vị trí cắt trong `1..W-1` rồi suy ra chiều rộng các viên (đảm bảo mọi tổng hàng bằng `W` và mọi viên `≥ 1`).

In [28]:
def random_wall(rng, max_w=14, max_n=8):
    W = rng.randint(1, max_w)
    n = rng.randint(1, max_n)
    wall = []
    for _ in range(n):
        cuts = sorted(rng.sample(range(1, W), rng.randint(0, W - 1))) if W > 1 else []
        pts = [0] + cuts + [W]
        wall.append([b - a for a, b in zip(pts, pts[1:])])
    return wall


rng = random.Random(554)
checked = 0
for _ in range(3000):
    wall = random_wall(rng)
    assert len({sum(r) for r in wall}) == 1 and all(x >= 1 for r in wall for x in r)
    expected = oracle_grid(wall)
    for name, f in solvers.items():
        got = f(wall)
        assert got == expected, (name, wall, got, expected)
    checked += 1
print("Ngẫu nhiên (seed 554):", checked, "tường, mọi cách đều khớp oracle lưới")

Ngẫu nhiên (seed 554): 3000 tường, mọi cách đều khớp oracle lưới


In [29]:
# Ngẫu nhiên cỡ vừa (không chạy brute/oracle theo tọa độ): đối chiếu Counter / sorted / heap / candidates
rng = random.Random(2026)
for _ in range(300):
    W = rng.randint(50, 500)
    n = rng.randint(1, 60)
    wall = []
    for _ in range(n):
        cuts = sorted(rng.sample(range(1, W), rng.randint(0, min(W - 1, 40))))
        pts = [0] + cuts + [W]
        wall.append([b - a for a, b in zip(pts, pts[1:])])
    results = {name: f(wall) for name, f in fast_solvers.items()}
    assert len(set(results.values())) == 1, (results, wall)
    assert results["counter"] == oracle_grid(wall)
print("Ngẫu nhiên cỡ vừa: OK")

Ngẫu nhiên cỡ vừa: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tính cả tổng tiền tố của viên cuối (mép phải)

Nếu dùng `for x in row` thay vì `for x in row[:-1]`, tổng tiền tố cuối của **mọi hàng** đều bằng `W`, nên `cnt[W] = n` và bảng đếm luôn "thấy" một vị trí chung mọi hàng → đáp án `0`. Nhưng `x = W` là mép phải, bị cấm.

In [30]:
def wrong_include_last(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row:               # SAI: gồm cả viên cuối -> s cuối = W (mép phải)
            s += x
            cnt[s] += 1
    return len(wall) - max(cnt.values(), default=0)

print("sai:", wrong_include_last([[1], [1], [1]]), "| đúng:", Solution().leastBricks([[1], [1], [1]]))
print("sai:", wrong_include_last(examples[0][0]), "| đúng:", Solution().leastBricks(examples[0][0]))
assert wrong_include_last([[1], [1], [1]]) == 0       # sai (đáp án đúng là 3)


# Đúng: bỏ viên cuối
def right_exclude_last(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    return len(wall) - max(cnt.values(), default=0)

print("đúng:", right_exclude_last([[1], [1], [1]]))

sai: 0 | đúng: 3
sai: 0 | đúng: 2
đúng: 3


### Lỗi 2 — `max()` trên bảng đếm rỗng

Khi **mọi hàng chỉ có một viên** (như ví dụ 2), không ai bỏ phiếu nên `cnt` rỗng. `max(cnt.values())` ném `ValueError: max() iterable argument is empty` và chương trình sập. Phải truyền `default=0`.

In [31]:
def wrong_empty_max(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    return len(wall) - max(cnt.values())      # SAI: cnt có thể rỗng

try:
    wrong_empty_max([[1], [1], [1]])
except ValueError as e:
    print("ValueError:", e)

print("đúng (default=0):", right_exclude_last([[1], [1], [1]]))

ValueError: max() iterable argument is empty
đúng (default=0): 3


### Lỗi 3 — Trả về `max(cnt.values())` thay vì phần bù `n - max`

`max(cnt.values())` là số hàng **không bị cắt** ở vị trí tốt nhất, còn đề hỏi số gạch **bị cắt**. Hai số này cộng lại bằng `n`. Ví dụ 1 cho thấy ngay: `max = 4` còn đáp án đúng là `2`.

In [32]:
def wrong_return_max(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    return max(cnt.values(), default=0)       # SAI: đây là số hàng không bị cắt

print("sai:", wrong_return_max(examples[0][0]), "| đúng:", Solution().leastBricks(examples[0][0]))
assert wrong_return_max(examples[0][0]) == 4 and Solution().leastBricks(examples[0][0]) == 2

sai: 4 | đúng: 2


### Lỗi 4 — Duyệt từng tọa độ khi chiều rộng rất lớn

Cách 1 (duyệt `x` từ `1` đến `W - 1`) nhìn đúng nhưng với `W` cỡ hàng tỷ sẽ chạy hàng giờ. Ta đếm **số vị trí phải thử** thay vì đo thời gian để thấy sự chênh lệch.

In [33]:
wall_huge = [[M, M], [M, M], [1, 2 * M - 1]]
W_huge = sum(wall_huge[0])
mach_vua = set()
for row in wall_huge:
    mach_vua.update(accumulate(row[:-1]))
print("chiều rộng W            =", W_huge)
print("brute phải thử số vị trí =", W_huge - 1)
print("chỉ cần xét mạch vữa     =", len(mach_vua), "vị trí:", sorted(mach_vua))
print("đáp án (Counter)         =", Solution().leastBricks(wall_huge))
assert Solution().leastBricks(wall_huge) == 1

chiều rộng W            = 4294967294
brute phải thử số vị trí = 4294967293
chỉ cần xét mạch vữa     = 2 vị trí: [1, 2147483647]
đáp án (Counter)         = 1


## 14. Mẹo & nhận dạng pattern

- **"Tìm vị trí/giá trị xuất hiện nhiều nhất trong nhiều danh sách"** thường là **bảng đếm** (`Counter`). Hãy nghĩ cách để *mỗi danh sách bỏ phiếu*, thay vì hỏi từng ứng viên.
- **"Đường/khoảng cắt qua ít thứ nhất"** thường đổi thành **"đi qua nhiều ranh giới nhất"**: cắt ít = trùng ranh giới nhiều. Công thức bù: `tổng - số lượt trùng`.
- **Chiều rộng cực lớn nhưng số phần tử nhỏ** → đừng duyệt theo tọa độ; hãy duyệt theo **các điểm đáng chú ý** (ở đây là các tổng tiền tố). Đây là tư tưởng "nén tọa độ".
- **Tổng tiền tố làm khóa của bảng băm** là một mẫu rất thường gặp, ví dụ *Subarray Sum Equals K* (LeetCode 560): đếm số lần xuất hiện tổng tiền tố.
- **Nhớ biên**: mép phải (viên cuối) bị loại; bảng đếm rỗng khi mọi hàng một viên.
- Nếu đề cho phép chỉ so sánh trong một hàng (ví dụ yêu cầu "vị trí"), hãy lưu thêm khóa `x` trong `Counter` (đã có sẵn) thay vì chỉ giữ giá trị lớn nhất.

## 15. Biến thể

### 15.1. Trả về cả vị trí đường thẳng tối ưu

Nếu đề yêu cầu in vị trí `x` (nhỏ nhất trong các vị trí tối ưu) cùng số gạch bị cắt: dùng luôn bảng `cnt`.

In [34]:
def best_line(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    if not cnt:
        return None, len(wall)               # không có mạch vữa: mọi vị trí đều cắt n viên
    best = max(cnt.values())
    x = min(k for k, v in cnt.items() if v == best)
    return x, len(wall) - best

print(best_line(examples[0][0]))     # (4, 2)
print(best_line([[1], [1], [1]]))    # (None, 3)
print(best_line([[2, 3], [2, 3]]))   # (2, 0)
assert best_line(examples[0][0]) == (4, 2)

(4, 2)
(None, 3)
(2, 0)


### 15.2. Có bao nhiêu vị trí cho kết quả tối ưu?

Đếm số khóa đạt `max` trong `Counter`. Ví dụ 1 chỉ có một vị trí (`x = 4`); tường `[[1,2],[2,1]]` có hai vị trí cùng cắt 1 viên (`x = 1` và `x = 2`).

In [35]:
def count_optimal_positions(wall):
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    if not cnt:
        return 0
    best = max(cnt.values())
    return sum(1 for v in cnt.values() if v == best)

print(count_optimal_positions(examples[0][0]))   # 1
print(count_optimal_positions([[1, 2], [2, 1]])) # 2
assert count_optimal_positions([[1, 2], [2, 1]]) == 2

1
2


### 15.3. Vẽ hai (hoặc `k`) đường thẳng

Nếu được vẽ `k` đường ở **các vị trí khác nhau** và tổng gạch bị cắt tính theo từng đường (một viên có thể bị hai đường cắt thì tính hai lần), thì mỗi đường độc lập: chọn `k` vị trí có phiếu cao nhất, đáp án `k · n - (tổng k phiếu cao nhất)` (điều kiện đủ nhiều vị trí hợp lệ). Đây chính là bài tập 5 (mục 17).

### 15.4. Các biến thể khác (chỉ mô tả)

- **Gạch có chiều cao khác nhau**: không còn "mỗi hàng bị cắt đúng 1 viên" nên phải gom theo từng đoạn cao; cần quét theo khoảng (sweep line) chứ không chỉ đếm.
- **Tường tròn / cuộn** (hai mép dính nhau): mép trái/phải trở thành một mạch vữa chung.
- **Cho phép sát mép**: đáp án luôn `0`, bài thành tầm thường, đó là lý do đề cấm.
- **Bài liên quan**: Subarray Sum Equals K (560), Contiguous Array (525), Subarray Sums Divisible by K (974), Minimum Number of Arrows to Burst Balloons (452).

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng tọa độ | duyệt mọi `x` trong `1..W-1`, tính số gạch bị cắt | `O(W · N)` | `O(1)` | chỉ để kiểm chứng trên tường rất nhỏ (làm oracle) |
| 2. Set + thử ứng viên | lập set mạch vữa mỗi hàng, thử từng ứng viên | `O(E · n + N)` | `O(N)` | dễ hiểu; chấp nhận khi `E · n` nhỏ; dễ TLE ở giới hạn đề |
| 3. Counter + tiền tố | mỗi hàng bỏ phiếu cho mạch vữa, lấy phiếu lớn nhất | `O(N)` | `O(E)` | **cách nên nộp**: ngắn, nhanh, ít lỗi |
| 4. Sắp xếp + groupby | gom mọi mạch vữa, sắp xếp, tìm cụm dài nhất | `O(N log N)` | `O(N)` | khi không muốn dùng bảng băm, hoặc muốn kết quả đã có thứ tự |
| 5. Heap gộp k danh sách | quét mạch vữa từ trái qua phải bằng min-heap | `O(N log n)` | `O(n)` | khi các danh sách đã sắp xếp sẵn và cần bộ nhớ nhỏ |

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(list(accumulate([1, 2, 3])))
print([1, 2, 3][:-1])
print(Counter(accumulate([2, 2, 1][:-1])), max([], default=-1))
```

In [36]:
# Bài 1: viết dự đoán của bạn vào các biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Với `wall = [[2,2,1],[1,4],[5],[3,2],[2,3]]`, hãy liệt kê mạch vữa bên trong của từng hàng, lập bảng đếm, và cho biết: vị trí `x` tốt nhất, số hàng chung mạch vữa tại đó, và đáp án của bài.

In [37]:
# Bài 2: điền kết quả tính tay
hand_best_x = None        # vị trí tốt nhất
hand_rows_sharing = None  # số hàng chung mạch vữa tại vị trí đó
hand_answer = None        # đáp án bài toán

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo khuôn mẫu Counter.

In [38]:
def exercise_3_least_bricks(wall: List[List[int]]) -> int:
    cnt = Counter()
    for row in wall:
        s = 0
        # TODO 1: duyệt các viên gạch của row TRỪ viên cuối
        #     TODO 2: cộng dồn chiều rộng vào s
        #     TODO 3: bỏ một phiếu cho vị trí s trong cnt
        pass
    # TODO 4: trả về số hàng trừ đi số phiếu lớn nhất (nhớ default=0)
    return None


# assert exercise_3_least_bricks([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 2
# assert exercise_3_least_bricks([[1], [1], [1]]) == 3
# assert exercise_3_least_bricks([[2, 3], [2, 3]]) == 0

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại các cách trên, hãy tự viết `exercise_4_least_bricks_sorted(wall)` theo cách **sắp xếp + gom nhóm**: gom mọi mạch vữa vào một danh sách, sắp xếp, tìm đoạn liên tiếp bằng nhau dài nhất (không dùng `Counter` và không dùng `groupby` cũng được: dùng một vòng `for` so với phần tử trước).

In [39]:
def exercise_4_least_bricks_sorted(wall: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_4_least_bricks_sorted([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 2
# assert exercise_4_least_bricks_sorted([[1], [1], [1]]) == 3
# assert exercise_4_least_bricks_sorted([[1, 2], [2, 1]]) == 1

### Bài 5 — Mini challenge: hai đường thẳng (khó)

Được vẽ **hai** đường thẳng đứng ở **hai vị trí nguyên khác nhau** `x ≠ y`, đều thỏa `0 < x, y < W`. Tổng số gạch bị cắt tính theo từng đường (viên bị cả hai đường cắt thì tính 2 lần). Viết `exercise_5_two_lines(wall)` trả về tổng nhỏ nhất. Giả sử `W ≥ 3` (luôn có ít nhất hai vị trí hợp lệ).

```python
exercise_5_two_lines([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) -> 5   # phiếu 4 và 3
exercise_5_two_lines([[3], [3]]) -> 4                                                              # không có mạch vữa: 2 * 2
exercise_5_two_lines([[1, 2], [2, 1]]) -> 2                                                        # hai vị trí 1 và 2, mỗi nơi cắt 1
```

In [40]:
def exercise_5_two_lines(wall: List[List[int]]) -> int:
    # TODO
    pass


# assert exercise_5_two_lines([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 5
# assert exercise_5_two_lines([[3], [3]]) == 4
# assert exercise_5_two_lines([[1, 2], [2, 1]]) == 2

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `accumulate` cộng dồn; `[:-1]` bỏ phần tử cuối.
- Mức 2: `accumulate([1,2,3])` cho `1, 3, 6`; `[2,2,1][:-1] = [2,2]` có tổng tiền tố `2, 4`.
- Mức 3: dòng 1 là `[1, 3, 6]`; dòng 2 là `[1, 2]`; dòng 3 in `Counter({2: 1, 4: 1}) -1`.

**Bài 2**
- Mức 1: bỏ viên cuối của mỗi hàng rồi cộng dồn: hàng 2 `[5]` không có mạch vữa.
- Mức 2: hàng 0 cho `2, 4`; hàng 1 cho `1`; hàng 3 cho `3`; hàng 4 cho `2`.
- Mức 3: `cnt = {1:1, 2:2, 3:1, 4:1}` → tốt nhất `x = 2` với 2 hàng → đáp án `5 - 2 = 3`.

**Bài 3**
- Mức 1: `for x in row[:-1]:` rồi `s += x`, `cnt[s] += 1`.
- Mức 2: đừng quên `s = 0` ở đầu mỗi hàng (đã có sẵn trong khung).
- Mức 3: `return len(wall) - max(cnt.values(), default=0)`.

**Bài 4**
- Mức 1: `edges = []`, `edges.extend(accumulate(row[:-1]))` cho mỗi hàng, rồi `edges.sort()`.
- Mức 2: duyệt `edges` giữ `run` (độ dài đoạn bằng nhau đang đếm) và `best`; nếu `edges[i] == edges[i-1]` thì `run += 1` ngược lại `run = 1`.
- Mức 3: cẩn thận danh sách rỗng: đặt `best = 0` ban đầu; kết quả `len(wall) - best`.

**Bài 5**
- Mức 1: mỗi đường độc lập nên chọn hai vị trí có phiếu cao nhất.
- Mức 2: `counts = sorted(cnt.values(), reverse=True)[:2]`; thiếu phiếu thì coi như `0` (vẫn có vị trí khác không mạch vữa nếu `W ≥ 3`).
- Mức 3: `2 * n - sum(counts)`.

## 19. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này. Mỗi cell dưới đây đều tự kiểm tra bằng `assert`.

### Đáp án Bài 1

In [41]:
print(list(accumulate([1, 2, 3])))
print([1, 2, 3][:-1])
print(Counter(accumulate([2, 2, 1][:-1])), max([], default=-1))

assert list(accumulate([1, 2, 3])) == [1, 3, 6]
assert [1, 2, 3][:-1] == [1, 2]
assert Counter(accumulate([2, 2, 1][:-1])) == Counter({2: 1, 4: 1})
assert max([], default=-1) == -1
print("Bài 1 OK")

[1, 3, 6]
[1, 2]
Counter({2: 1, 4: 1}) -1
Bài 1 OK


### Đáp án Bài 2

Mạch vữa: hàng 0 → `2, 4`; hàng 1 → `1`; hàng 2 → (không có); hàng 3 → `3`; hàng 4 → `2`. Bảng đếm `{1:1, 2:2, 3:1, 4:1}`; tốt nhất `x = 2` (2 hàng), đáp án `5 - 2 = 3`.

In [42]:
wall_ex2 = [[2, 2, 1], [1, 4], [5], [3, 2], [2, 3]]
cnt_ex2 = Counter()
for row in wall_ex2:
    s = 0
    for x in row[:-1]:
        s += x
        cnt_ex2[s] += 1
print(dict(sorted(cnt_ex2.items())))
best_x = max(cnt_ex2, key=lambda k: cnt_ex2[k])
print("x tốt nhất =", best_x, "| số hàng =", cnt_ex2[best_x], "| đáp án =", len(wall_ex2) - cnt_ex2[best_x])

assert dict(sorted(cnt_ex2.items())) == {1: 1, 2: 2, 3: 1, 4: 1}
assert (best_x, cnt_ex2[best_x], len(wall_ex2) - cnt_ex2[best_x]) == (2, 2, 3)
assert Solution().leastBricks(wall_ex2) == oracle_grid(wall_ex2) == 3
print("Bài 2 OK")

{1: 1, 2: 2, 3: 1, 4: 1}
x tốt nhất = 2 | số hàng = 2 | đáp án = 3
Bài 2 OK


### Đáp án Bài 3

In [43]:
def exercise_3_solution(wall: List[List[int]]) -> int:
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    return len(wall) - max(cnt.values(), default=0)


assert exercise_3_solution([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 2
assert exercise_3_solution([[1], [1], [1]]) == 3
assert exercise_3_solution([[2, 3], [2, 3]]) == 0
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [44]:
def exercise_4_solution(wall: List[List[int]]) -> int:
    edges = []
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            edges.append(s)
    edges.sort()
    best = 0
    run = 0
    for i, e in enumerate(edges):
        if i > 0 and e == edges[i - 1]:
            run += 1
        else:
            run = 1
        best = max(best, run)
    return len(wall) - best


assert exercise_4_solution([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 2
assert exercise_4_solution([[1], [1], [1]]) == 3
assert exercise_4_solution([[1, 2], [2, 1]]) == 1
rng = random.Random(4)
for _ in range(500):
    wall = random_wall(rng)
    assert exercise_4_solution(wall) == oracle_grid(wall)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [45]:
def exercise_5_solution(wall: List[List[int]]) -> int:
    cnt = Counter()
    for row in wall:
        s = 0
        for x in row[:-1]:
            s += x
            cnt[s] += 1
    top2 = sorted(cnt.values(), reverse=True)[:2]     # thiếu phiếu thì coi như 0
    return 2 * len(wall) - sum(top2)


def oracle_two_lines(wall):
    W = sum(wall[0])
    best = None
    for x in range(1, W):
        for y in range(x + 1, W):
            total = crossed_at(wall, x) + crossed_at(wall, y)
            best = total if best is None else min(best, total)
    return best


assert exercise_5_solution([[1, 2, 2, 1], [3, 1, 2], [1, 3, 2], [2, 4], [3, 1, 2], [1, 3, 1, 1]]) == 5
assert exercise_5_solution([[3], [3]]) == 4
assert exercise_5_solution([[1, 2], [2, 1]]) == 2
rng = random.Random(5)
tested = 0
while tested < 800:
    wall = random_wall(rng, max_w=12)
    if sum(wall[0]) < 3:
        continue                                      # cần ít nhất hai vị trí hợp lệ
    assert exercise_5_solution(wall) == oracle_two_lines(wall), wall
    tested += 1
print("Bài 5 OK (", tested, "tường ngẫu nhiên khớp oracle )")

Bài 5 OK ( 800 tường ngẫu nhiên khớp oracle )


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def leastBricks(self, wall: List[List[int]]) -> int:
        cnt = Counter()
        for row in wall:
            s = 0
            for x in row[:-1]:          # bỏ viên cuối: mép phải của tường
                s += x                  # tổng tiền tố = vị trí mạch vữa
                cnt[s] += 1             # hàng này bỏ 1 phiếu cho vị trí s
        return len(wall) - max(cnt.values(), default=0)
```

Câu thần chú: **"mỗi hàng bỏ phiếu cho mạch vữa (trừ viên cuối); đáp án = số hàng - phiếu lớn nhất."**

### Checklist trước khi submit

- [ ] Dùng `row[:-1]` (bỏ viên cuối) để không đếm mép phải.
- [ ] `s = 0` đặt **trong** vòng lặp hàng (reset cho mỗi hàng).
- [ ] `max(cnt.values(), default=0)` để không lỗi khi mọi hàng một viên.
- [ ] Trả về `len(wall) - max_phiếu`, không phải `max_phiếu`.
- [ ] Không duyệt theo tọa độ `x` (chiều rộng có thể hàng tỷ).
- [ ] Test: ví dụ đề, `[[1],[1],[1]]`, một hàng nhiều viên (đáp án 0), giá trị `2^31 - 1`.
- [ ] Skeleton đúng: `class Solution`, method `leastBricks(self, wall: List[List[int]]) -> int`.